In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

In [2]:
ROOT = Path.cwd()
if not (ROOT / 'ASU_IFT103_OS_Resource_Monitor.xlsx').exists():
    raise FileNotFoundError('Start Jupyter in this repository directory.')
file_path = ROOT / 'ASU_IFT103_OS_Resource_Monitor.xlsx'
df = pd.read_excel(file_path)
df.head()

         Date   Time  Process Name   PID  CPU %  Memory MB Process Type
0  2025-04-09  08:00    chrome.exe  1000   34.7       2553     User App
1  2025-04-09  08:00       systemd  1001   13.9       2591       System
2  2025-04-09  08:00    python.exe  1002    6.6        120     Dev Tool
3  2025-04-09  08:00  explorer.exe  1003   28.3       1281     User App
4  2025-04-09  08:00   svchost.exe  1004   21.9       2819       System


In [3]:
conn = sqlite3.connect(ROOT / 'os_metrics.db')
df.to_sql('process_metrics', conn, if_exists='replace', index=False)

150


In [4]:
query1 = '''
SELECT "Process Name", AVG("CPU %") AS avg_cpu
FROM process_metrics
GROUP BY "Process Name"
ORDER BY avg_cpu DESC
LIMIT 5;
'''
pd.read_sql(query1, conn)

  Process Name    avg_cpu
0     zoom.exe  21.653333
1       mysqld  19.820000
2   chrome.exe  19.500000
3   vscode.exe  19.073333
4  svchost.exe  18.273333


In [5]:
query2 = '''
SELECT Date, AVG("Memory MB") AS avg_memory
FROM process_metrics
GROUP BY Date
'''
pd.read_sql(query2, conn)

         Date  avg_memory
0  2025-04-09     2271.22
1  2025-04-10     2140.02
2  2025-04-11     2501.00


In [6]:
query3 = '''
SELECT "Process Type", COUNT(*) as total
FROM process_metrics
GROUP BY "Process Type";
'''
pd.read_sql(query3, conn)

  Process Type  total
0     Dev Tool     45
1       System     45
2     User App     60


In [7]:
query4 = '''
SELECT *
FROM process_metrics
WHERE "CPU %" > 30;
'''
df_high_cpu = pd.read_sql(query4, conn)
df_high_cpu


          Date   Time  Process Name   PID  CPU %  Memory MB Process Type
0   2025-04-09  08:00    chrome.exe  1000   34.7       2553     User App
1   2025-04-09  08:00      code.exe  1005   32.9       1054     Dev Tool
2   2025-04-09  10:00   svchost.exe  1014   32.2       1015       System
3   2025-04-09  10:00      zoom.exe  1016   34.5       4658     User App
4   2025-04-09  12:00  explorer.exe  1023   34.2       2296     User App
..         ...    ...           ...   ...    ...        ...          ...
19  2025-04-11  08:00      zoom.exe  1106   30.8        409     User App
20  2025-04-11  10:00    python.exe  1112   34.1       3474     Dev Tool
21  2025-04-11  14:00  explorer.exe  1133   33.0        272     User App
22  2025-04-11  16:00        mysqld  1147   30.9       4555       System
23  2025-04-11  16:00  onedrive.exe  1149   34.9        364     User App


In [8]:
export_path = ROOT / 'High_CPU_Processes.xlsx'
df_high_cpu.to_excel(export_path, index=False)